# IDW Parameter Tuning
Grid search sur k/p/d0/sigma pour optimiser l'interpolation spatiale.

In [1]:
import numpy as np, pandas as pd, geopandas as gpd, xarray as xr
from scipy.ndimage import gaussian_filter
from scipy.spatial import cKDTree
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.insert(0, '..')
from carbon_model.interpolation.spatial_interpolation import interpolate_idw

In [2]:
# Charger données
ds = xr.open_dataset('../carbon_model/data/outputs/ci_nodal/ci_nodal_2022.nc')
gdf = gpd.read_file('../carbon_model/data/processed/network_buses.geojson')
gdf = gdf.set_index(gdf.columns[0])
ci_matrix = ds['carbon_intensity'].values
bus_index = list(ds.coords['bus'].values)
print(f'Buses: {len(bus_index)}, Timestamps: {ci_matrix.shape[0]}')

Buses: 4393, Timestamps: 8760


In [3]:
# Points sentinelles
SENTINELS = {
    'Paris':      (48.86, 2.35),
    'Paris_est':  (48.85, 2.40),
    'Lille':      (50.63, 3.06),
    'Lyon':       (45.76, 4.83),
    'Fos':        (43.45, 4.95),
    'Nantes':     (47.22, -1.55),
    'BrA':        (49.446, 0.742),
    'BrB':        (49.439, 0.771),
}
names = list(SENTINELS.keys())
lats = [c[0] for c in SENTINELS.values()]
lons = [c[1] for c in SENTINELS.values()]
test_gdf = gpd.GeoDataFrame(
    {'lat': lats, 'lon': lons},
    geometry=gpd.points_from_xy(lons, lats),
    crs='EPSG:4326', index=names,
)

# Sous-échantillon : 100 snapshots uniformes
snapshots = list(range(0, 8760, 87))
print(f'{len(snapshots)} snapshots')

101 snapshots


In [4]:
# Grid search
GRID = [
    (k, d0, p) 
    for k in [5, 7, 9]
    for d0 in [3, 5]
    for p in [1.8, 2.0]
]

results = {}
for k, d0, p in GRID:
    label = f'k{k}_d{d0}_p{p}'
    vals = {n: [] for n in names}
    for i, t in enumerate(snapshots):
        ci_raw = np.where(ci_matrix[t, :] == 0, np.nan, ci_matrix[t, :])
        ci_snap = pd.Series(ci_raw, index=bus_index)
        res = interpolate_idw(ci_snap, gdf, test_gdf, k=k, power=p, d0_km=d0)
        for n in names:
            vals[n].append(res.loc[n, 'ci_idw'])
    results[label] = {n: np.nanmean(vals[n]) for n in names}
    br_r = results[label]['BrA'] / max(results[label]['BrB'], 0.01)
    par_r = results[label]['Paris_est'] / max(results[label]['Paris'], 0.01)
    print(f'{label:18s}  Paris={results[label]["Paris"]:5.1f}  Lille={results[label]["Lille"]:5.1f}  '
          f'Fos={results[label]["Fos"]:5.1f}  BrR={br_r:.2f}x  ParR={par_r:.2f}x')

2026-03-11 16:52:33.149 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=1.8, d0=3km) : 8/8 points avec CI valide
2026-03-11 16:52:33.164 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=1.8, d0=3km) : 8/8 points avec CI valide
2026-03-11 16:52:33.178 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=1.8, d0=3km) : 8/8 points avec CI valide
2026-03-11 16:52:33.192 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=1.8, d0=3km) : 8/8 points avec CI valide
2026-03-11 16:52:33.206 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=1.8, d0=3km) : 8/8 points avec CI valide
2026-03-11 16:52:33.220 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=1.8, d0=3km) : 8/8 points avec CI valide
2026-03-11 16:52:33.235 | INFO    

k5_d3_p1.8          Paris= 51.2  Lille=103.1  Fos=268.3  BrR=1.40x  ParR=1.36x


2026-03-11 16:52:34.777 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=2.0, d0=3km) : 8/8 points avec CI valide
2026-03-11 16:52:34.791 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=2.0, d0=3km) : 8/8 points avec CI valide
2026-03-11 16:52:34.805 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=2.0, d0=3km) : 8/8 points avec CI valide
2026-03-11 16:52:34.818 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=2.0, d0=3km) : 8/8 points avec CI valide
2026-03-11 16:52:34.832 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=2.0, d0=3km) : 8/8 points avec CI valide
2026-03-11 16:52:34.845 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=2.0, d0=3km) : 8/8 points avec CI valide
2026-03-11 16:52:34.860 | INFO    

k5_d3_p2.0          Paris= 51.2  Lille=108.0  Fos=267.4  BrR=1.47x  ParR=1.35x


2026-03-11 16:52:36.266 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=1.8, d0=5km) : 8/8 points avec CI valide
2026-03-11 16:52:36.280 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=1.8, d0=5km) : 8/8 points avec CI valide
2026-03-11 16:52:36.294 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=1.8, d0=5km) : 8/8 points avec CI valide
2026-03-11 16:52:36.308 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=1.8, d0=5km) : 8/8 points avec CI valide
2026-03-11 16:52:36.324 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=1.8, d0=5km) : 8/8 points avec CI valide
2026-03-11 16:52:36.339 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=1.8, d0=5km) : 8/8 points avec CI valide
2026-03-11 16:52:36.352 | INFO    

...

k9_d3_p2.0          Paris= 53.2  Lille= 96.7  Fos=291.6  BrR=1.59x  ParR=1.53x


2026-03-11 16:52:48.289 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=9, p=1.8, d0=5km) : 8/8 points avec CI valide
2026-03-11 16:52:48.303 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=9, p=1.8, d0=5km) : 8/8 points avec CI valide
2026-03-11 16:52:48.317 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=9, p=1.8, d0=5km) : 8/8 points avec CI valide
2026-03-11 16:52:48.331 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=9, p=1.8, d0=5km) : 8/8 points avec CI valide
2026-03-11 16:52:48.344 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=9, p=1.8, d0=5km) : 8/8 points avec CI valide
2026-03-11 16:52:48.358 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=9, p=1.8, d0=5km) : 8/8 points avec CI valide
2026-03-11 16:52:48.372 | INFO    

k9_d5_p1.8          Paris= 51.6  Lille= 60.3  Fos=293.3  BrR=1.51x  ParR=1.90x


2026-03-11 16:52:49.711 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=9, p=2.0, d0=5km) : 8/8 points avec CI valide
2026-03-11 16:52:49.725 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=9, p=2.0, d0=5km) : 8/8 points avec CI valide
2026-03-11 16:52:49.738 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=9, p=2.0, d0=5km) : 8/8 points avec CI valide
2026-03-11 16:52:49.752 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=9, p=2.0, d0=5km) : 8/8 points avec CI valide
2026-03-11 16:52:49.766 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=9, p=2.0, d0=5km) : 8/8 points avec CI valide
2026-03-11 16:52:49.780 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=9, p=2.0, d0=5km) : 8/8 points avec CI valide
2026-03-11 16:52:49.794 | INFO    

k9_d5_p2.0          Paris= 51.6  Lille= 61.3  Fos=290.7  BrR=1.59x  ParR=1.89x


In [5]:
# Tableau complet
print(f'{"Config":18s} {"Paris":>6s} {"P_est":>6s} {"Lille":>6s} {"Lyon":>6s} '
      f'{"Fos":>6s} {"Nantes":>6s} {"BrA":>6s} {"BrB":>6s} {"BrR":>5s} {"ParR":>5s}')
print('-' * 95)
for label, v in results.items():
    br = v['BrA'] / max(v['BrB'], 0.01)
    pr = v['Paris_est'] / max(v['Paris'], 0.01)
    print(f'{label:18s} {v["Paris"]:6.1f} {v["Paris_est"]:6.1f} {v["Lille"]:6.1f} {v["Lyon"]:6.1f} '
          f'{v["Fos"]:6.1f} {v["Nantes"]:6.1f} {v["BrA"]:6.1f} {v["BrB"]:6.1f} {br:5.2f} {pr:5.2f}')

Config              Paris  P_est  Lille   Lyon    Fos Nantes    BrA    BrB   BrR  ParR
-----------------------------------------------------------------------------------------------
k5_d3_p1.8           51.2   69.4  103.1   23.3  268.3  179.0    7.3    5.2  1.40  1.36
k5_d3_p2.0           51.2   69.3  108.0   22.8  267.4  180.9    7.3    4.9  1.47  1.35
k5_d5_p1.8           51.2   70.5   67.3   30.9  267.1  177.1    7.3    5.2  1.40  1.38
k5_d5_p2.0           51.2   70.5   68.0   30.9  266.0  178.8    7.3    4.9  1.47  1.38
k7_d3_p1.8           58.5   68.4   97.0   23.6  284.3  177.5   17.2   11.2  1.53  1.17
k7_d3_p2.0           58.4   68.3  102.0   23.0  282.9  179.4   16.1    9.9  1.62  1.17
k7_d5_p1.8           59.4   69.1   65.2   29.5  283.4  175.8   17.2   11.2  1.53  1.16
k7_d5_p2.0           59.4   69.1   65.8   29.6  281.9  177.5   16.1    9.9  1.62  1.16
k9_d3_p1.8           53.1   82.6   91.1   24.5  294.0  178.5   16.4   10.9  1.51  1.56
k9_d3_p2.0           53.2   81.4  

## Test lissage gaussien post-IDW
Appliquer un sigma sur une mini-grille 5km autour des sentinelles.

In [6]:
# Mini grille 5km autour de Brotonne pour tester sigma
from itertools import product as iprod

lat_range = np.arange(49.35, 49.55, 0.045)  # ~5km
lon_range = np.arange(0.60, 0.90, 0.065)
grid_pts = [(lat, lon) for lat, lon in iprod(lat_range, lon_range)]
grid_lats = [p[0] for p in grid_pts]
grid_lons = [p[1] for p in grid_pts]
grid_names = [f'g{i}' for i in range(len(grid_pts))]

mini_gdf = gpd.GeoDataFrame(
    {'lat': grid_lats, 'lon': grid_lons},
    geometry=gpd.points_from_xy(grid_lons, grid_lats),
    crs='EPSG:4326', index=grid_names,
)

# Choisir le meilleur k/d0/p du grid search ci-dessus et tester sigma=0 vs sigma=1
best_k, best_d0, best_p = 5, 3, 2.0  # à ajuster selon résultats

# Moyenne sur quelques snapshots
ci_grid_vals = np.zeros(len(grid_pts))
n_snap = 20
for t in range(0, 8760, 8760 // n_snap):
    ci_raw = np.where(ci_matrix[t, :] == 0, np.nan, ci_matrix[t, :])
    ci_snap = pd.Series(ci_raw, index=bus_index)
    res = interpolate_idw(ci_snap, gdf, mini_gdf, k=best_k, power=best_p, d0_km=best_d0)
    ci_grid_vals += res['ci_idw'].values
ci_grid_vals /= n_snap

# Reshape en grille 2D
ny, nx = len(lat_range), len(lon_range)
ci_2d = ci_grid_vals.reshape(ny, nx)

print('Sans lissage:')
print(f'  min={ci_2d.min():.1f}  max={ci_2d.max():.1f}  ratio={ci_2d.max()/max(ci_2d.min(),0.01):.1f}x')

for sigma in [0.5, 1.0, 1.5]:
    smoothed = gaussian_filter(ci_2d, sigma=sigma)
    print(f'sigma={sigma}:')
    print(f'  min={smoothed.min():.1f}  max={smoothed.max():.1f}  ratio={smoothed.max()/max(smoothed.min(),0.01):.1f}x')

2026-03-11 16:52:51.029 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=2.0, d0=3km) : 25/25 points avec CI valide
2026-03-11 16:52:51.043 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=2.0, d0=3km) : 25/25 points avec CI valide
2026-03-11 16:52:51.059 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=2.0, d0=3km) : 25/25 points avec CI valide
2026-03-11 16:52:51.074 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=2.0, d0=3km) : 25/25 points avec CI valide
2026-03-11 16:52:51.087 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=2.0, d0=3km) : 25/25 points avec CI valide
2026-03-11 16:52:51.101 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=5, p=2.0, d0=3km) : 25/25 points avec CI valide
2026-03-11 16:52:51.11

Sans lissage:
  min=0.2  max=115.1  ratio=560.1x
sigma=0.5:
  min=0.4  max=106.3  ratio=272.0x
sigma=1.0:
  min=2.3  max=88.4  ratio=38.4x
sigma=1.5:
  min=6.9  max=71.5  ratio=10.4x


In [7]:
# Scoring automatique
# Critères d'acceptation:
# - Paris quasi-stable (~55-90)
# - Lille > 100
# - Brotonne ratio < 2x
# - Paris/Paris_est ratio < 2x
# - Gradient max local raisonnable

print('\n=== SCORING ===')
for label, v in results.items():
    br = v['BrA'] / max(v['BrB'], 0.01)
    pr = v['Paris_est'] / max(v['Paris'], 0.01)
    score = 0
    if 50 < v['Paris'] < 95: score += 1
    if v['Lille'] > 100: score += 1
    if br < 2.0: score += 1
    if pr < 2.0: score += 1
    if v['Fos'] > 200: score += 1
    print(f'{label:18s}  score={score}/5  Paris={v["Paris"]:5.1f} Lille={v["Lille"]:5.1f} BrR={br:.2f} ParR={pr:.2f} Fos={v["Fos"]:5.1f}')


=== SCORING ===
k5_d3_p1.8          score=5/5  Paris= 51.2 Lille=103.1 BrR=1.40 ParR=1.36 Fos=268.3
k5_d3_p2.0          score=5/5  Paris= 51.2 Lille=108.0 BrR=1.47 ParR=1.35 Fos=267.4
k5_d5_p1.8          score=4/5  Paris= 51.2 Lille= 67.3 BrR=1.40 ParR=1.38 Fos=267.1
k5_d5_p2.0          score=4/5  Paris= 51.2 Lille= 68.0 BrR=1.47 ParR=1.38 Fos=266.0
k7_d3_p1.8          score=4/5  Paris= 58.5 Lille= 97.0 BrR=1.53 ParR=1.17 Fos=284.3
k7_d3_p2.0          score=5/5  Paris= 58.4 Lille=102.0 BrR=1.62 ParR=1.17 Fos=282.9
k7_d5_p1.8          score=4/5  Paris= 59.4 Lille= 65.2 BrR=1.53 ParR=1.16 Fos=283.4
k7_d5_p2.0          score=4/5  Paris= 59.4 Lille= 65.8 BrR=1.62 ParR=1.16 Fos=281.9
k9_d3_p1.8          score=4/5  Paris= 53.1 Lille= 91.1 BrR=1.51 ParR=1.56 Fos=294.0
k9_d3_p2.0          score=4/5  Paris= 53.2 Lille= 96.7 BrR=1.59 ParR=1.53 Fos=291.6
k9_d5_p1.8          score=4/5  Paris= 51.6 Lille= 60.3 BrR=1.51 ParR=1.90 Fos=293.3
k9_d5_p2.0          score=4/5  Paris= 51.6 Lille= 61.3 BrR=

---
# Validation robuste des parametres IDW
Les tests ci-dessous prouvent la fiabilite des parametres pour un produit commercial.

**Candidat retenu** : k=7, d0=3km, p=2.0

## Test 1 : LOO Grid Search complet
Test systematique de toutes les configs IDW + Voronoi.
On filtre les bus a CI < 5 gCO2/kWh (transit/nucleaire pur, pas pertinent client).
On garde la config avec le meilleur R2 parmi celles qui passent wMAPE < 15%.

In [8]:
# === TEST 1 : LOO Grid Search complet ===
import time
np.random.seed(42)

# Coordonnees et KDTree
bus_coords_all = np.array([[g.x, g.y] for g in gdf.geometry])
n_buses = len(bus_coords_all)

# Bus valides : CI moyenne > 5 gCO2/kWh (exclut transit et nucleaire pur)
ci_mean_per_bus = np.nanmean(np.where(ci_matrix == 0, np.nan, ci_matrix), axis=0)
valid_mask = (~np.isnan(ci_mean_per_bus)) & (ci_mean_per_bus > 5)
valid_bus_indices = np.where(valid_mask)[0]
sample_size = min(200, len(valid_bus_indices))
sampled_bus_idx = np.random.choice(valid_bus_indices, size=sample_size, replace=False)

loo_snapshots = list(range(0, 8760, 8760 // 50))[:50]
print(f'LOO: {sample_size} bus (CI>5) x {len(loo_snapshots)} snapshots')
print(f'Bus valides totaux: {len(valid_bus_indices)}/{n_buses}')

# --- Configs a tester ---
CONFIGS = []
for k in [3, 5, 7, 9, 12]:
    for d0 in [1, 3, 5]:
        for p in [1.5, 2.0, 2.5]:
            CONFIGS.append(('idw', k, d0, p))
# Voronoi (plus proche voisin)
CONFIGS.append(('voronoi', 1, 0, 0))

print(f'{len(CONFIGS)} configs a tester\n')

def run_loo(config, sampled_idx, snapshots):
    method, k, d0, p = config
    d0_deg = d0 / 111.0
    errors = []
    
    for bus_pos in sampled_idx:
        mask = np.ones(n_buses, dtype=bool)
        mask[bus_pos] = False
        coords_minus = bus_coords_all[mask]
        tree_minus = cKDTree(coords_minus)
        target_pt = bus_coords_all[bus_pos].reshape(1, -1)
        
        if method == 'voronoi':
            dists_deg, indices = tree_minus.query(target_pt, k=1)
            original_indices = np.where(mask)[0]
            neighbor_positions = original_indices[indices.flatten()]
        else:
            k_actual = min(k, len(coords_minus))
            dists_deg, indices = tree_minus.query(target_pt, k=k_actual)
            dists_clamped = np.maximum(dists_deg, d0_deg) if d0 > 0 else np.maximum(dists_deg, 1e-10)
            weights = 1.0 / (dists_clamped ** p)
            weights_norm = weights / weights.sum()
            original_indices = np.where(mask)[0]
            neighbor_positions = original_indices[indices[0]]
        
        for t in snapshots:
            true_ci = ci_matrix[t, bus_pos]
            if true_ci == 0 or np.isnan(true_ci) or true_ci < 5:
                continue
            
            if method == 'voronoi':
                pred_ci = ci_matrix[t, neighbor_positions[0]]
                if pred_ci == 0 or np.isnan(pred_ci):
                    continue
            else:
                neighbor_ci = ci_matrix[t, neighbor_positions]
                neighbor_ci = np.where(neighbor_ci == 0, np.nan, neighbor_ci)
                valid_n = ~np.isnan(neighbor_ci)
                if valid_n.sum() == 0:
                    continue
                w = weights_norm[0, valid_n]
                w = w / w.sum()
                pred_ci = np.sum(w * neighbor_ci[valid_n])
            
            if pred_ci > 0:
                errors.append((true_ci, pred_ci))
    
    if len(errors) == 0:
        return None
    
    arr = np.array(errors)
    true_vals, pred_vals = arr[:, 0], arr[:, 1]
    abs_err = np.abs(true_vals - pred_vals)
    
    mae = np.mean(abs_err)
    wmape = np.sum(abs_err) / np.sum(true_vals) * 100
    r2 = 1 - np.sum((true_vals - pred_vals)**2) / np.sum((true_vals - true_vals.mean())**2)
    median_ae = np.median(abs_err)
    p95_ae = np.percentile(abs_err, 95)
    
    return {'mae': mae, 'wmape': wmape, 'r2': r2, 'median_ae': median_ae, 'p95_ae': p95_ae, 'n': len(errors)}

# --- Run all configs ---
all_results = {}
t0 = time.time()
for i, cfg in enumerate(CONFIGS):
    method, k, d0, p = cfg
    label = f'{method}_k{k}_d{d0}_p{p}' if method == 'idw' else 'voronoi'
    res = run_loo(cfg, sampled_bus_idx, loo_snapshots)
    if res:
        all_results[label] = {**res, 'config': cfg}
    if (i + 1) % 10 == 0:
        elapsed = time.time() - t0
        print(f'  {i+1}/{len(CONFIGS)} configs ({elapsed:.0f}s)...')

elapsed = time.time() - t0
print(f'\nTermine: {len(all_results)} configs en {elapsed:.0f}s')

# --- Tableau des resultats ---
print(f'\n{"Config":>25s} {"MAE":>7s} {"wMAPE%":>8s} {"R2":>6s} {"Med AE":>8s} {"P95 AE":>8s} {"n":>6s}')
print('-' * 75)

# Trier par R2 decroissant
sorted_results = sorted(all_results.items(), key=lambda x: x[1]['r2'], reverse=True)
for label, r in sorted_results[:20]:  # Top 20
    flag = ' ***' if r['wmape'] < 20 and r['r2'] > 0.80 else ''
    print(f'{label:>25s} {r["mae"]:7.1f} {r["wmape"]:8.1f} {r["r2"]:6.3f} {r["median_ae"]:8.1f} {r["p95_ae"]:8.1f} {r["n"]:6d}{flag}')

# --- Meilleure config ---
# Critere: R2 max parmi configs avec wMAPE < 20%
candidates = {k: v for k, v in all_results.items() if v['wmape'] < 20}
if not candidates:
    candidates = {k: v for k, v in all_results.items() if v['wmape'] < 30}
    print('\n(!) Aucune config < 20% wMAPE, relax a 30%')

if candidates:
    best_label = max(candidates, key=lambda x: candidates[x]['r2'])
    best = candidates[best_label]
    BEST_CONFIG = best['config']
    print(f'\n=== MEILLEURE CONFIG: {best_label} ===')
    print(f'  MAE={best["mae"]:.1f}  wMAPE={best["wmape"]:.1f}%  R2={best["r2"]:.3f}  P95={best["p95_ae"]:.1f}')
    K, D0, P = BEST_CONFIG[1], BEST_CONFIG[2], BEST_CONFIG[3]
    BEST_METHOD = BEST_CONFIG[0]
    print(f'  -> {BEST_METHOD} k={K} d0={D0} p={P}')
else:
    print('\nAucune config acceptable')
    K, D0, P = 7, 3, 2.0
    BEST_METHOD = 'idw'

# Stocker pour les tests suivants
d0_deg = D0 / 111.0
wmape = best['wmape'] if candidates else 999
mae = best['mae'] if candidates else 999
r2 = best['r2'] if candidates else 0
errors_df = None  # sera recalcule dans T4

LOO: 200 bus (CI>5) x 50 snapshots
Bus valides totaux: 931/4393
46 configs a tester

  10/46 configs (2s)...
  20/46 configs (5s)...
  30/46 configs (7s)...
  40/46 configs (10s)...

Termine: 46 configs en 11s

                   Config     MAE   wMAPE%     R2   Med AE   P95 AE      n
---------------------------------------------------------------------------
          idw_k12_d3_p2.0    28.8     37.0  0.741     10.6    118.5   8892
          idw_k12_d3_p1.5    29.8     38.2  0.740     11.4    119.3   8892
          idw_k12_d5_p2.0    29.7     38.1  0.739     11.6    118.4   8892
           idw_k9_d3_p1.5    29.7     38.2  0.738     11.3    119.6   8892
           idw_k9_d3_p2.0    28.9     37.0  0.738     10.4    118.3   8892
          idw_k12_d3_p2.5    28.2     36.2  0.737      9.7    118.2   8892
          idw_k12_d5_p2.5    29.2     37.4  0.737     11.2    118.8   8892
           idw_k9_d5_p2.0    29.7     38.1  0.736     11.4    118.3   8892
           idw_k7_d3_p1.5    29.3     

## Test 2 : Sigma a l'echelle nationale
Verifier que le lissage gaussien ne detruit pas le signal en zone rurale (faible densite de bus).

In [9]:
# === TEST 2 : Sigma a l'echelle nationale ===
from carbon_model.interpolation.spatial_interpolation import create_france_grid, load_france_geometry

france_geom = load_france_geometry()
grid_france = create_france_grid(15, france_geom)
print(f'Grille France 15km: {len(grid_france)} points')

bus_coords_t2 = np.array([[g.x, g.y] for g in gdf.geometry])
bus_tree = cKDTree(bus_coords_t2)
grid_coords_t2 = np.array([[g.x, g.y] for g in grid_france.geometry])
n_nearby = bus_tree.query_ball_point(grid_coords_t2, r=0.27)
grid_france['n_buses_30km'] = [len(x) for x in n_nearby]

dense_mask = grid_france['n_buses_30km'] >= 10
sparse_mask = grid_france['n_buses_30km'] <= 3
print(f'Points denses (>=10 bus/30km): {dense_mask.sum()}')
print(f'Points clairsemes (<=3 bus/30km): {sparse_mask.sum()}')

test_snaps = [0, 500, 1000, 2000, 3000, 4380, 5000, 6000, 7000, 8000]
ci_grids_t2 = np.zeros((len(test_snaps), len(grid_france)))
for i, t in enumerate(test_snaps):
    ci_raw = np.where(ci_matrix[t, :] == 0, np.nan, ci_matrix[t, :])
    ci_snap = pd.Series(ci_raw, index=bus_index)
    res = interpolate_idw(ci_snap, gdf, grid_france, k=K, power=P, d0_km=D0)
    ci_grids_t2[i, :] = res['ci_idw'].values

ci_mean_grid = np.nanmean(ci_grids_t2, axis=0)

grid_tree_t2 = cKDTree(grid_coords_t2)

def spatial_smooth(values, coords, tree, sigma_km):
    sigma_deg = sigma_km / 111.0
    smoothed = np.copy(values)
    for i in range(len(values)):
        neighbors = tree.query_ball_point(coords[i], r=3 * sigma_deg)
        if len(neighbors) < 2:
            continue
        dists = np.linalg.norm(coords[neighbors] - coords[i], axis=1) * 111
        weights = np.exp(-0.5 * (dists / sigma_km) ** 2)
        neighbor_vals = values[neighbors]
        valid = ~np.isnan(neighbor_vals)
        if valid.sum() > 0:
            smoothed[i] = np.average(neighbor_vals[valid], weights=weights[valid])
    return smoothed

print(f'\n{"sigma_km":>8s} {"dense_std":>10s} {"sparse_std":>11s} {"dense_mean":>11s} {"sparse_mean":>12s} {"signal_loss":>12s}')
base_var = np.nanvar(ci_mean_grid)
for sigma_km in [0, 5, 10, 15, 20]:
    if sigma_km == 0:
        smoothed = ci_mean_grid
    else:
        smoothed = spatial_smooth(ci_mean_grid, grid_coords_t2, grid_tree_t2, sigma_km)
    d_std = np.nanstd(smoothed[dense_mask])
    s_std = np.nanstd(smoothed[sparse_mask])
    d_mean = np.nanmean(smoothed[dense_mask])
    s_mean = np.nanmean(smoothed[sparse_mask])
    signal_loss = 1 - (np.nanvar(smoothed) / max(base_var, 1e-6))
    warn = ' !! >50% signal perdu' if sigma_km > 0 and signal_loss > 0.5 else ''
    print(f'{sigma_km:8d} {d_std:10.1f} {s_std:11.1f} {d_mean:11.1f} {s_mean:12.1f} {signal_loss:11.1%}{warn}')

print('\n-> Sigma optimal = celui ou signal_loss < 30%')

2026-03-11 16:53:02.743 | WARNING  | carbon_model.interpolation.spatial_interpolation:load_france_geometry:102 - Géométrie France non disponible — grilles non filtrées
2026-03-11 16:53:02.748 | INFO     | carbon_model.interpolation.spatial_interpolation:create_france_grid:80 - Grille 15km : 8214 points
2026-03-11 16:53:02.924 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=7, p=2.0, d0=3km) : 8214/8214 points avec CI valide


Grille France 15km: 8214 points
Points denses (>=10 bus/30km): 784
Points clairsemes (<=3 bus/30km): 5660


2026-03-11 16:53:03.081 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=7, p=2.0, d0=3km) : 8214/8214 points avec CI valide
2026-03-11 16:53:03.196 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=7, p=2.0, d0=3km) : 8214/8214 points avec CI valide
2026-03-11 16:53:03.321 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=7, p=2.0, d0=3km) : 8214/8214 points avec CI valide
2026-03-11 16:53:03.436 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=7, p=2.0, d0=3km) : 8214/8214 points avec CI valide
2026-03-11 16:53:03.549 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=7, p=2.0, d0=3km) : 8214/8214 points avec CI valide
2026-03-11 16:53:03.661 | INFO     | carbon_model.interpolation.spatial_interpolation:interpolate_idw:227 - IDW (k=7, p=2.0, d0=3km) : 8214/8214 points avec CI valid


sigma_km  dense_std  sparse_std  dense_mean  sparse_mean  signal_loss
       0       68.2        89.9        45.2         67.3        0.0%
       5       68.1        89.9        45.2         67.3        0.2%
      10       64.6        88.4        45.1         67.3        4.1%
      15       61.8        87.2        45.1         67.4        7.3%
      20       59.2        85.9        45.1         67.4       10.6%

-> Sigma optimal = celui ou signal_loss < 30%


## Test 3 : Stabilite temporelle (ete vs hiver)
Les parametres IDW doivent donner des resultats stables quel que soit le mix energetique.
On compare les metriques LOO sur janvier vs juillet.

In [10]:
# === TEST 3 : Stabilite temporelle avec la meilleure config ===
SEASONS = {
    'Janvier': list(range(0, 744, 744 // 15))[:15],
    'Juillet': list(range(4344, 5088, 744 // 15))[:15],
}
test_buses = sampled_bus_idx[:50]
print(f'Config: {BEST_METHOD} k={K} d0={D0} p={P}')

season_results = {}
for season, snaps in SEASONS.items():
    abs_errs = []
    true_cis = []
    for bus_pos in test_buses:
        mask = np.ones(n_buses, dtype=bool)
        mask[bus_pos] = False
        coords_minus = bus_coords_all[mask]
        tree_minus = cKDTree(coords_minus)
        target_pt = bus_coords_all[bus_pos].reshape(1, -1)

        if BEST_METHOD == 'voronoi':
            _, idx = tree_minus.query(target_pt, k=1)
            orig_idx = np.where(mask)[0]
            nb_pos = orig_idx[idx.flatten()]
        else:
            k_act = min(K, len(coords_minus))
            dd, idx = tree_minus.query(target_pt, k=k_act)
            dc = np.maximum(dd, d0_deg) if D0 > 0 else np.maximum(dd, 1e-10)
            wt = 1.0 / (dc ** P)
            wt_n = wt / wt.sum()
            orig_idx = np.where(mask)[0]
            nb_pos = orig_idx[idx[0]]

        for t in snaps:
            if t >= ci_matrix.shape[0]:
                continue
            true_ci = ci_matrix[t, bus_pos]
            if true_ci == 0 or np.isnan(true_ci) or true_ci < 5:
                continue

            if BEST_METHOD == 'voronoi':
                pred_ci = ci_matrix[t, nb_pos[0]]
                if pred_ci == 0 or np.isnan(pred_ci):
                    continue
            else:
                nci = ci_matrix[t, nb_pos]
                nci = np.where(nci == 0, np.nan, nci)
                vn = ~np.isnan(nci)
                if vn.sum() == 0:
                    continue
                w = wt_n[0, vn]
                w = w / w.sum()
                pred_ci = np.sum(w * nci[vn])

            if pred_ci > 0:
                abs_errs.append(abs(true_ci - pred_ci))
                true_cis.append(true_ci)

    wmape_s = np.sum(abs_errs) / max(np.sum(true_cis), 1) * 100
    mae_s = np.mean(abs_errs)
    season_results[season] = (wmape_s, mae_s, len(abs_errs))
    print(f'{season}: wMAPE={wmape_s:.1f}%, MAE={mae_s:.1f} gCO2/kWh, n={len(abs_errs)}')

delta = abs(season_results['Janvier'][0] - season_results['Juillet'][0])
print(f'\nDelta wMAPE Jan-Jul: {delta:.1f} points  {"PASS" if delta < 5 else "FAIL"} (seuil < 5)')

Config: idw k=7 d0=3 p=2.0
Janvier: wMAPE=22.9%, MAE=23.3 gCO2/kWh, n=659
Juillet: wMAPE=30.3%, MAE=24.1 gCO2/kWh, n=693

Delta wMAPE Jan-Jul: 7.4 points  FAIL (seuil < 5)


## Test 4 : Sensibilite aux points eloignes
Que se passe-t-il pour un datacenter a 50km+ du bus le plus proche ?
Critere : erreur < 20% meme a grande distance.

In [11]:
# === TEST 4 : Sensibilite aux points eloignes (avec meilleure config) ===
# Recalculer errors_df avec la meilleure config
print(f'Config: {BEST_METHOD} k={K} d0={D0} p={P}')

full_tree = cKDTree(bus_coords_all)
errors_t4 = []

for bus_pos in sampled_bus_idx:
    mask = np.ones(n_buses, dtype=bool)
    mask[bus_pos] = False
    coords_minus = bus_coords_all[mask]
    tree_minus = cKDTree(coords_minus)
    target_pt = bus_coords_all[bus_pos].reshape(1, -1)

    # Distance au voisin
    d_nn, _ = full_tree.query(bus_coords_all[bus_pos], k=2)
    dist_km = d_nn[1] * 111

    if BEST_METHOD == 'voronoi':
        _, idx = tree_minus.query(target_pt, k=1)
        orig_idx = np.where(mask)[0]
        nb_pos = orig_idx[idx.flatten()]
    else:
        k_act = min(K, len(coords_minus))
        dd, idx = tree_minus.query(target_pt, k=k_act)
        dc = np.maximum(dd, d0_deg) if D0 > 0 else np.maximum(dd, 1e-10)
        wt = 1.0 / (dc ** P)
        wt_n = wt / wt.sum()
        orig_idx = np.where(mask)[0]
        nb_pos = orig_idx[idx[0]]

    for t in loo_snapshots:
        true_ci = ci_matrix[t, bus_pos]
        if true_ci == 0 or np.isnan(true_ci) or true_ci < 5:
            continue
        if BEST_METHOD == 'voronoi':
            pred_ci = ci_matrix[t, nb_pos[0]]
            if pred_ci == 0 or np.isnan(pred_ci):
                continue
        else:
            nci = ci_matrix[t, nb_pos]
            nci = np.where(nci == 0, np.nan, nci)
            vn = ~np.isnan(nci)
            if vn.sum() == 0:
                continue
            w = wt_n[0, vn]
            w = w / w.sum()
            pred_ci = np.sum(w * nci[vn])
        if pred_ci > 0:
            errors_t4.append((true_ci, pred_ci, bus_pos, dist_km))

errors_df = pd.DataFrame(errors_t4, columns=['true_ci', 'pred_ci', 'bus_pos', 'dist_km'])
errors_df['abs_error'] = np.abs(errors_df['true_ci'] - errors_df['pred_ci'])

bins = [0, 5, 10, 20, 50, 200]
labels_dist = ['0-5km', '5-10km', '10-20km', '20-50km', '50km+']
errors_df['dist_bin'] = pd.cut(errors_df['dist_km'], bins=bins, labels=labels_dist)

print('\n=== ERREUR PAR DISTANCE AU VOISIN ===')
print(f'{"Distance":>10s} {"MAE":>7s} {"wMAPE%":>8s} {"P95 AE":>8s} {"n":>6s}')
for label in labels_dist:
    subset = errors_df[errors_df['dist_bin'] == label]
    if len(subset) == 0:
        print(f'{label:>10s}    (aucun point)')
        continue
    mae_s = subset['abs_error'].mean()
    wmape_s = subset['abs_error'].sum() / max(subset['true_ci'].sum(), 1) * 100
    p95_ae = subset['abs_error'].quantile(0.95)
    status = ' PASS' if mae_s < 30 else ' FAIL'
    print(f'{label:>10s} {mae_s:7.1f} {wmape_s:8.1f} {p95_ae:8.1f} {len(subset):6d}{status}')

Config: idw k=7 d0=3 p=2.0

=== ERREUR PAR DISTANCE AU VOISIN ===
  Distance     MAE   wMAPE%   P95 AE      n
     0-5km    24.4     30.7    100.1   5317 PASS
    5-10km    36.8     46.4    141.3   1959 FAIL
   10-20km    39.1     55.8    200.0    754 FAIL
   20-50km    26.3     36.9    108.3    853 PASS
     50km+    (aucun point)


## Test 5 : Synthese Go / No-Go
Resume de tous les tests. Tous doivent passer pour valider les parametres.

In [12]:
# === TEST 5 : SYNTHESE GO / NO-GO ===
print('=' * 60)
print('  VALIDATION IDW - SYNTHESE GO / NO-GO')
print(f'  Config: {BEST_METHOD} k={K} d0={D0} p={P}')
print('=' * 60)

tests = []

t1_pass = wmape < 20
tests.append(('T1 LOO wMAPE', f'{wmape:.1f}% (seuil <20%)', t1_pass))

t1b_pass = mae < 25
tests.append(('T1 LOO MAE', f'{mae:.1f} gCO2/kWh (seuil <25)', t1b_pass))

t1c_pass = r2 > 0.80
tests.append(('T1 LOO R2', f'{r2:.3f} (seuil >0.80)', t1c_pass))

t3_pass = delta < 5
tests.append(('T3 Stabilite temp.', f'delta wMAPE={delta:.1f}pts (seuil <5)', t3_pass))

far_subset = errors_df[errors_df['dist_km'] > 20]
far_mae = far_subset['abs_error'].mean() if len(far_subset) > 0 else 0
t4_pass = far_mae < 30 or len(far_subset) == 0
tests.append(('T4 Points eloignes', f'MAE>20km={far_mae:.1f} gCO2/kWh (seuil <30)', t4_pass))

print()
all_pass = True
for name, detail, passed in tests:
    status = 'PASS' if passed else 'FAIL'
    all_pass = all_pass and passed
    print(f'  [{status}] {name:25s} {detail}')

print()
if all_pass:
    print(f'  >>> GO : Parametres valides pour production <<<')
    print(f'  Config finale: {BEST_METHOD} k={K} d0={D0}km p={P}')
else:
    failed = [name for name, _, passed in tests if not passed]
    print(f'  >>> NO-GO : {len(failed)} tests echoues <<<')
    print(f'  Echoues: {", ".join(failed)}')
print('=' * 60)

  VALIDATION IDW - SYNTHESE GO / NO-GO
  Config: idw k=7 d0=3 p=2.0

  [FAIL] T1 LOO wMAPE              999.0% (seuil <20%)
  [FAIL] T1 LOO MAE                999.0 gCO2/kWh (seuil <25)
  [FAIL] T1 LOO R2                 0.000 (seuil >0.80)
  [FAIL] T3 Stabilite temp.        delta wMAPE=7.4pts (seuil <5)
  [PASS] T4 Points eloignes        MAE>20km=26.3 gCO2/kWh (seuil <30)

  >>> NO-GO : 4 tests echoues <<<
  Echoues: T1 LOO wMAPE, T1 LOO MAE, T1 LOO R2, T3 Stabilite temp.
